In [6]:
import os
import time
import random
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
from thop import profile
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import all routing modules (use the improved versions)
from routing_smoe import SMoELayer
from routing_micro import MICROMoELayer
from routing_expert_choice import ExpertChoiceMoELayer
from routing_adaptive import AdaptiveDynamicMoELayer
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Running on: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🌱 Seed set to {seed}")

✅ Running on: cuda


In [7]:
class Config:
    MODEL_NAME = "vinai/phobert-large"
    TRAIN_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\train.csv"
    VAL_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\validation.csv"
    TEST_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    BATCH_SIZE = 12
    LR = 1.5e-5
    EPOCHS = 10
    PATIENCE = 3
    SEEDS = [42, 123, 999]
    NUM_EXPERTS = 8
    ROUTING_TYPES_TO_TEST = ["expert_choice", "smoe", "micro", "adaptive", "deepseek"]
    CAPACITY_FACTOR = 1.5
    CHECKPOINT_DIR = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\checkpoints_phobert_multi"
    RESULTS_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\results_phobert_multi.csv"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

In [8]:
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class NLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        print(f"Pre-tokenizing {len(df)} samples...")
        self.encodings = tokenizer(
            df['premise'].astype(str).tolist(),
            df['hypothesis'].astype(str).tolist(),
            max_length=max_len, padding='max_length', truncation=True, return_tensors='pt'
        )
        print("✅ Tokenization done!")

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

train_loader = DataLoader(NLIDataset(df_train, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(NLIDataset(df_val, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(NLIDataset(df_test, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)

Pre-tokenizing 8012 samples...
✅ Tokenization done!
Pre-tokenizing 1000 samples...
✅ Tokenization done!
Pre-tokenizing 1000 samples...
✅ Tokenization done!


In [ ]:
# ========================= UTILS & ARCHITECTURE =========================
class CheckpointManager:
    def __init__(self, model, model_name="moe"):
        self.model = model
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pth")
        self.results_path = Config.RESULTS_CSV
        
        # Tạo file CSV và Header nếu chưa tồn tại
        if not os.path.exists(self.results_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Val_Acc", "Val_F1", "GFlops", "Runtime_ms", "VRAM_MB", "Entropy","Expert_Usage"])
            df.to_csv(self.results_path, index=False)

    def save_checkpoint(self, epoch, val_f1, is_best=False):
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'best_val_f1': val_f1
        }
        if is_best: 
            torch.save(state, self.best_checkpoint_path)

    def log_results(self, row):
        # Cơ chế Append trực tiếp vào file CSV
        df = pd.read_csv(self.results_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.results_path, index=False)

def load_balancing_loss(router_output, num_experts):
    """Universal Load Balancing Loss"""
    if router_output is None:
        return torch.tensor(0.0, device=device)
    if isinstance(router_output, tuple):
        router_output = router_output[0]
    probs = torch.softmax(router_output, dim=-1)
    expert_probs = probs.mean(dim=0)
    ideal = torch.ones(num_experts, device=probs.device) / num_experts
    return torch.mean((expert_probs - ideal) ** 2)

def calculate_routing_metrics(model):
    """Tính toán Entropy và xuất mảng phân phối tần suất của Expert"""
    metrics = {"entropy": 0.0, "expert_usage_distribution": None}
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            # Lấy xác suất trung bình phân bổ cho từng expert
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = probs.mean(dim=0).cpu().numpy().tolist()
            
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception:
        pass
    return metrics

class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size

        for name, param in self.backbone.named_parameters():
            if 'encoder.layer' in name and int(name.split('.')[2]) < 12:
                param.requires_grad = False

        self.routing_type = routing_type

        if routing_type == "expert_choice":
            self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)
        elif routing_type == "smoe":
            self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "micro":
            self.moe_layer = MICROMoELayer(hidden_size)
        elif routing_type == "adaptive":
            self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "deepseek":
            self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=2, num_routed_experts=config.NUM_EXPERTS-2)
        else:
            self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)

        self.pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, config.NUM_LABELS)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        moe_output = self.moe_layer(sequence_output)
        pooled = self.pooling(moe_output, attention_mask)
        return self.classifier(pooled)

In [ ]:
# ========================= MEGA PIPELINE =========================
all_test_results = []

for seed in Config.SEEDS:
    set_seed(seed)
    for routing_type in Config.ROUTING_TYPES_TO_TEST:
        print(f"\n{'='*70}")
        print(f"🚀 SEED {seed} | ROUTING: {routing_type.upper()} | {Config.NUM_EXPERTS} Experts")
        print(f"{'='*70}")

        model = UnifiedMoENLI(Config(), routing_type).to(device)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
        total_steps = len(train_loader) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
        scaler = GradScaler()
        criterion = nn.CrossEntropyLoss()
        
        # Khởi tạo CheckpointManager cho mỗi Seed và Routing
        model_name = f"phobert_large_{routing_type}_seed{seed}"
        checkpoint_manager = CheckpointManager(model, model_name=model_name)

        # Tính GFLOPS 1 lần đầu tiên
        dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
        gflops = (macs * 2) / 1e9
        print(f"📊 Estimated GFLOPS: {gflops:.2f}")

        best_f1 = 0.0
        patience_counter = 0

        for epoch in range(Config.EPOCHS):
            model.train()
            train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1} [{routing_type}]", leave=False)
            for batch in train_iterator:
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    logits = model(ids, mask)
                    ce_loss = criterion(logits, labels)
                    router_out = None
                    if hasattr(model.moe_layer, 'router'):
                        try:
                            router_out = model.moe_layer.router(ids)
                        except:
                            pass
                    lb_loss = load_balancing_loss(router_out, Config.NUM_EXPERTS)
                    loss = ce_loss + 0.01 * lb_loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                train_iterator.set_postfix(loss=f"{loss.item():.4f}")

            # ================= VALIDATION =================
            model.eval()
            val_preds, val_labels = [], []
            
            torch.cuda.synchronize()
            start_time = time.time()
            
            with torch.inference_mode():
                for batch in val_loader:
                    ids = batch['input_ids'].to(device, non_blocking=True)
                    mask = batch['attention_mask'].to(device, non_blocking=True)
                    labels = batch['labels'].to(device, non_blocking=True)
                    with autocast():
                        logits = model(ids, mask)
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    
            torch.cuda.synchronize()
            runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
            vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            routing_stats = calculate_routing_metrics(model)
            entropy_val = routing_stats["entropy"]
            expert_dist = routing_stats["expert_usage_distribution"]

            print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
            
            # Lưu lịch sử (Append CSV) sau MỖI EPOCH
            checkpoint_manager.log_results({
                "Seed": seed, "Epoch": epoch+1, "Routing": routing_type, 
                "Val_Acc": val_acc, "Val_F1": val_f1, "GFlops": gflops, 
                "Runtime_ms": runtime_ms, "VRAM_MB": vram_mb, "Entropy": entropy_val,
                "Expert_Usage": str(expert_dist)
            })

            # Early Stopping & Checkpoint Save
            is_best = val_f1 > best_f1
            if is_best:
                best_f1 = val_f1
                patience_counter = 0
                print("✨ Val F1 cải thiện, lưu Best Checkpoint.")
            else:
                patience_counter += 1
                if patience_counter >= Config.PATIENCE:
                    print("🛑 Early stopping kích hoạt!")
                    break
                    
            checkpoint_manager.save_checkpoint(epoch, best_f1, is_best)

        # ================= TEST EVALUATION =================
        print(f"\n📥 Loading best checkpoint cho {routing_type.upper()}...")
        state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
        model.load_state_dict(state["model_state"])
        model.eval()
        
        test_preds, test_labels = [], []
        with torch.inference_mode():
            for batch in test_loader:
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                with autocast():
                    logits = model(ids, mask)
                test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                test_labels.extend(labels.cpu().numpy())

        test_acc = accuracy_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds, average='macro')
        print(f"🎯 TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f} ({routing_type})\n")

        all_test_results.append({
            "Seed": seed, "Routing": routing_type, "Test_Acc": test_acc, 
            "Test_F1": test_f1, "GFLOPS": gflops
        })

        # Dọn dẹp RAM/VRAM
        del model, optimizer, scheduler, checkpoint_manager
        torch.cuda.empty_cache()
        gc.collect()

print("✅ Hoàn tất huấn luyện toàn bộ kiến trúc!")
# Tuỳ chọn ghi thêm bảng tổng hợp Test Set
pd.DataFrame(all_test_results).to_csv(Config.RESULTS_CSV.replace(".csv", "_test_summary.csv"), index=False)

🌱 Seed set to 42

🚀 SEED 42 | ROUTING: EXPERT_CHOICE | 8 Experts
📊 Estimated GFLOPS: 161.71


KeyboardInterrupt: 